# Final Comparison — 3 Models
**Kết hợp kết quả từ Baseline 1, Baseline 2, và Proposed**  
Chạy notebook này SAU KHI đã train xong cả 3 model và **upload các artifact** sau vào dataset của notebook này (hoặc chúng nằm trong `/kaggle/working`):

- `evaluation_baseline1.json`, `evaluation_baseline2.json`, `evaluation_proposed.json`
- `predictions_baseline1.npz`, `predictions_baseline2.npz`, `predictions_proposed.npz`
- `training_logs_baseline1.json`, `training_logs_baseline2.json`, `training_logs_proposed.json`
- `dataset_stats_iscx.json`, `dataset_stats_mendeley.json`, `dataset_stats_proposed.json`

Toàn bộ biểu đồ (ROC, confusion matrix, training curves, data distribution) đều **vẽ từ predictions thật**, không phải minh họa.

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

In [ ]:
# ── Load results + predictions ──
OUT_DIR  = Path('/kaggle/working')
FIG_DIR  = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIRS = [OUT_DIR, Path('/kaggle/input'), OUT_DIR / 'models']

def find(name):
    for d in DATA_DIRS:
        if not d.exists():
            continue
        hits = list(d.rglob(name))
        if hits:
            return hits[0]
    return None

def load_json(name, default=None):
    p = find(name)
    if p is not None:
        print(f'Loaded: {p}')
        return json.load(open(p))
    print(f'MISSING: {name}')
    return default

def load_npz(name):
    p = find(name)
    if p is not None:
        print(f'Loaded: {p}')
        return np.load(p, allow_pickle=True)
    print(f'MISSING: {name}')
    return None

eval_b1 = load_json('evaluation_baseline1.json', {})
eval_b2 = load_json('evaluation_baseline2.json', {})
eval_pr = load_json('evaluation_proposed.json', {})
npz_b1  = load_npz('predictions_baseline1.npz')
npz_b2  = load_npz('predictions_baseline2.npz')
npz_pr  = load_npz('predictions_proposed.npz')
logs_b1 = load_json('training_logs_baseline1.json', [])
logs_b2 = load_json('training_logs_baseline2.json', [])
logs_pr = load_json('training_logs_proposed.json', [])
stats_b1 = load_json('dataset_stats_iscx.json', {})
stats_b2 = load_json('dataset_stats_mendeley.json', {})
stats_pr = load_json('dataset_stats_proposed.json', {})

In [ ]:
# ── Bảng kết quả ──
names = ['accuracy', 'precision', 'recall', 'f1', 'auc']
model_labels = ['Baseline 1\n(ISCX)', 'Baseline 2\n(Mendeley URL)', 'Proposed\n(Full)']
model_colors = ['#1f77b4', '#e41a1c', '#33a02c']
results = [eval_b1, eval_b2, eval_pr]

print('\n' + '='*95)
header = f'{"Model":<35}'
for n in names:
    header += f' {n.upper():<16}'
print(header)
print('='*95)
for r, label in zip(results, model_labels):
    line = f'{r.get("model","")[:34]:<35}'
    for n in names:
        v = r.get(n, 0.0); s = r.get(f'{n}_std', 0.0)
        line += f' {v:.4f}+-{s:.4f}  '
    print(line)
print('='*95)

print('\n=== IMPROVEMENT: Proposed vs Baseline 2 ===')
for n in names:
    delta = eval_pr.get(n, 0.0) - eval_b2.get(n, 0.0)
    print(f'  {n.upper():10s}: {("+" if delta >= 0 else "")}{delta:.4f}')

print('\n=== IMPROVEMENT: Proposed vs Baseline 1 ===')
for n in names:
    delta = eval_pr.get(n, 0.0) - eval_b1.get(n, 0.0)
    print(f'  {n.upper():10s}: {("+" if delta >= 0 else "")}{delta:.4f}')

In [ ]:
# ══════════════════════════════════════════
# FIGURE 1: Grouped Bar Chart
# ══════════════════════════════════════════
x = np.arange(len(names))
w = 0.25
fig, ax = plt.subplots(figsize=(11, 5.5))
for i, r in enumerate(results):
    means = [r.get(n, 0.0) for n in names]
    errs  = [r.get(f'{n}_std', 0.0) for n in names]
    ax.bar(x + (i-1)*w, means, w, yerr=errs, capsize=3,
           label=model_labels[i], color=model_colors[i], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([n.capitalize() for n in names])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('3-Model Comparison (Mean +- Std)')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0.5, color='gray', ls='--', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'compare_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════
# FIGURE 2: Real ROC Curves from saved predictions
# ══════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7, 6))
notes = []
if npz_b1 is not None:
    p = npz_b1['preds']; l = npz_b1['labels']
    fpr, tpr, _ = roc_curve(l, p)
    ax.plot(fpr, tpr, color=model_colors[0], lw=2,
            label=f'Baseline 1 (AUC={roc_auc_score(l, p):.4f})')
    notes.append('B1: 5-fold CV (ISCX)')
if npz_b2 is not None:
    p = npz_b2['test_preds_mean']; l = npz_b2['test_labels']
    fpr, tpr, _ = roc_curve(l, p)
    ax.plot(fpr, tpr, color=model_colors[1], lw=2,
            label=f'Baseline 2 (AUC={roc_auc_score(l, p):.4f})')
    notes.append('B2: held-out test (Mendeley)')
if npz_pr is not None:
    p = npz_pr['test_preds_mean']; l = npz_pr['test_labels']
    fpr, tpr, _ = roc_curve(l, p)
    ax.plot(fpr, tpr, color=model_colors[2], lw=2,
            label=f'Proposed (AUC={roc_auc_score(l, p):.4f})')
    notes.append('Proposed: held-out test (Mendeley)')
ax.plot([0,1],[0,1], 'gray', lw=1, alpha=0.5)
ax.set_title('ROC Comparison (from saved predictions)\n' + ' | '.join(notes), fontsize=11)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'compare_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════
# FIGURE 3: Confusion Matrices
# ══════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
if npz_b1 is not None:
    p = npz_b1['preds']; l = npz_b1['labels']
    sns.heatmap(confusion_matrix(l, (p >= 0.5).astype(int)), annot=True, fmt='d',
                cmap='Blues', ax=axes[0], cbar=False,
                xticklabels=['Benign','Phishing'], yticklabels=['Benign','Phishing'])
    axes[0].set_title('Baseline 1 (ISCX CV)')
if npz_b2 is not None:
    p = npz_b2['test_preds_mean']; l = npz_b2['test_labels']
    sns.heatmap(confusion_matrix(l, (p >= 0.5).astype(int)), annot=True, fmt='d',
                cmap='Oranges', ax=axes[1], cbar=False,
                xticklabels=['Benign','Phishing'], yticklabels=['Benign','Phishing'])
    axes[1].set_title('Baseline 2 (Mendeley test)')
if npz_pr is not None:
    p = npz_pr['test_preds_mean']; l = npz_pr['test_labels']
    sns.heatmap(confusion_matrix(l, (p >= 0.5).astype(int)), annot=True, fmt='d',
                cmap='Greens', ax=axes[2], cbar=False,
                xticklabels=['Benign','Phishing'], yticklabels=['Benign','Phishing'])
    axes[2].set_title('Proposed (Mendeley test)')
for ax in axes:
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(FIG_DIR / 'compare_cm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════
# FIGURE 4: Training Curves (mean across folds)
# ══════════════════════════════════════════
def mean_curve(logs):
    if not logs:
        return None
    max_ep = max(len(h['epochs']) for h in logs)
    out = []
    for ep in range(1, max_ep + 1):
        rows = [h['epochs'][ep-1] for h in logs if len(h['epochs']) >= ep]
        out.append({'epoch': ep,
                    'train_loss': float(np.mean([r['train_loss'] for r in rows])),
                    'val_auc': float(np.mean([r['val_auc'] for r in rows])),
                    'val_f1': float(np.mean([r['val_f1'] for r in rows]))})
    return out

curves = [('Baseline 1', mean_curve(logs_b1), '#1f77b4'),
          ('Baseline 2', mean_curve(logs_b2), '#e41a1c'),
          ('Proposed',   mean_curve(logs_pr), '#33a02c')]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, (name, cv, color) in zip(axes, curves):
    if cv is None:
        ax.axis('off'); ax.set_title(name); continue
    e = [r['epoch'] for r in cv]
    ax.plot(e, [r['train_loss'] for r in cv], 'o-', color='#d62728', lw=1.4, ms=3, label='Train loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss', color='#d62728')
    ax.tick_params(axis='y', labelcolor='#d62728')
    ax2 = ax.twinx()
    ax2.plot(e, [r['val_auc'] for r in cv], 's-', color=color, lw=1.4, ms=3, label='Val AUC')
    ax2.plot(e, [r['val_f1'] for r in cv], 'd-', color='#7b3294', lw=1.4, ms=3, label='Val F1')
    ax2.set_ylim(0, 1); ax2.set_ylabel('Score')
    ax.set_title(f'{name} — Training Curves')
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=8, loc='center left', bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.savefig(FIG_DIR / 'compare_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════
# FIGURE 5: Dataset Distribution
# ══════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# ISCX
if stats_b1:
    counts = [stats_b1.get('n_benign', 0), stats_b1.get('n_phishing', 0)]
    bars = axes[0].bar(['Benign','Phishing'], counts, color=['#2ca02c','#d62728'], alpha=0.85)
    axes[0].set_title(f"ISCX-URL2016 (n={stats_b1.get('n_samples', 0):,})")
    for b, v in zip(bars, counts):
        axes[0].text(b.get_x()+b.get_width()/2, v, f'{v:,}', ha='center', va='bottom', fontsize=9)

# Mendeley full + sampled
if stats_b2:
    nf = [stats_b2.get('n_full_benign', 0), stats_b2.get('n_full_phishing', 0)]
    ns = [stats_b2.get('n_benign', 0), stats_b2.get('n_phishing', 0)]
    w = 0.38; xs = np.arange(2)
    axes[1].bar(xs - w/2, nf, w, label='Full', color=['#2ca02c','#d62728'], alpha=0.45)
    axes[1].bar(xs + w/2, ns, w, label='Sampled', color=['#2ca02c','#d62728'], alpha=0.9)
    axes[1].set_xticks(xs); axes[1].set_xticklabels(['Benign','Phishing'])
    axes[1].set_title(f"Mendeley (full={stats_b2.get('n_full', 0):,}, sampled={stats_b2.get('n_samples', 0):,})")
    axes[1].legend(fontsize=8)

# Proposed HTML coverage
if stats_pr:
    vals = [stats_pr.get('html_found', 0), stats_pr.get('html_missing', 0)]
    bars = axes[2].bar(['HTML found','HTML missing'], vals, color=['#1f78b4','#a6cee3'], alpha=0.85)
    axes[2].set_title(f"Proposed HTML Coverage (n={stats_pr.get('n_samples', 0):,})")
    for b, v in zip(bars, vals):
        axes[2].text(b.get_x()+b.get_width()/2, v, f'{v:,}', ha='center', va='bottom', fontsize=9)

for ax in axes:
    ax.set_ylabel('Samples')
plt.tight_layout()
plt.savefig(FIG_DIR / 'compare_dist.png', dpi=150, bbox_inches='tight')
plt.show()

---
### Kết luận
- **Baseline 1:** TabTransformer chỉ trên 29 features (ISCX-URL2016, 5-fold CV)
- **Baseline 2:** TabTransformer chỉ trên 12 URL features (Mendeley, held-out test)
- **Proposed:** Gated Fusion (URL + ModernBERT text + DOM) (Mendeley, held-out test)
- ROC / Confusion Matrix vẽ từ predictions thật của từng model.

Download từ Output tab:
- `figures/compare_bar.png`
- `figures/compare_roc.png`
- `figures/compare_cm.png`
- `figures/compare_curves.png`
- `figures/compare_dist.png`